<a href="https://colab.research.google.com/github/ShreyIND/ML/blob/main/multi_output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import layers,models,optimizers,losses

In [1]:
import zipfile
zipref=zipfile.ZipFile("/content/archive (2).zip")
zipref.extractall("/content")
zipref.close()

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np
import pandas as pd

In [4]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [6]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [7]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [8]:
df

,age,gender,img
0,67,1,67_1_0_20170120224807800.jpg.chip.jpg
1,53,1,53_1_0_20170110160643297.jpg.chip.jpg
2,69,1,69_1_0_20170110141630303.jpg.chip.jpg
3,78,0,78_0_0_20170111210454101.jpg.chip.jpg
4,30,1,30_1_0_20170117103601281.jpg.chip.jpg
...,...,...,...
23703,33,1,33_1_0_20170111182452825.jpg.chip.jpg
23704,45,0,45_0_2_20170117183529862.jpg.chip.jpg
23705,1,0,1_0_4_20161221202008129.jpg.chip.jpg
23706,49,0,49_0_3_20170104214709125.jpg.chip.jpg


In [10]:
df.shape

(23708, 3)

In [11]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [12]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [13]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [14]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [15]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [17]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [33]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [ ]:
def multi_output_generator(generator):
    for x, y_list in generator:
        yield x, {'age': y_list[0], 'gender': y_list[1]}

wrapped_train_generator = multi_output_generator(train_generator)
wrapped_test_generator = multi_output_generator(test_generator)

model.fit(
    wrapped_train_generator,
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=wrapped_test_generator,
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step - age_loss: 14.6519 - age_mae: 14.6519 - gender_accuracy: 0.5228 - gender_loss: 0.6933 - loss: 83.2887